<a href="https://colab.research.google.com/github/darshanlahamage/Adaptive-Adversarial-Augmentation/blob/main/notebooks/v1_pgd_latent_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PGD Latent Baseline — CIFAR-100 / CIFAR-100-C

In [1]:
import os, math, time, random, json, urllib.request, tarfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.fft as fft_module
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
def get_device():
    d = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'[*] Device: {d}')
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'    GPU : {p.name}')
        print(f'    VRAM: {p.total_memory/1e9:.1f} GB')
    return d

device = get_device()

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

set_seed(42)
print('[*] Seed 42')

def ensure_dir(p): Path(p).mkdir(parents=True, exist_ok=True)


[*] Device: cuda
    GPU : Tesla T4
    VRAM: 15.6 GB
[*] Seed 42


In [ ]:
def compute_ece(probs, correct, n_bins=15):
    """Expected Calibration Error."""
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece, n = 0.0, probs.shape[0]
    for i in range(n_bins):
        m = (probs >= edges[i]) & (probs < edges[i+1])
        if m.sum() == 0: continue
        ece += (m.sum()/n) * abs(correct[m].mean() - probs[m].mean())
    return float(ece)


def compute_failure_diversity(features, preds, trues, n_clusters=10, pca_dim=64):
    """Normalised entropy over K-Means clusters of failure feature vectors."""
    mask = preds != trues
    if mask.sum() == 0: return 0.0, 0
    X = features[mask]
    n = X.shape[0]
    if n <= 2: return 0.0, int(n)
    Xr  = PCA(n_components=min(pca_dim, X.shape[1]), random_state=0).fit_transform(X)
    k   = min(n_clusters, max(2, n // 10))
    lbl = KMeans(n_clusters=k, random_state=0, n_init=10).fit(Xr).labels_
    cnt = np.bincount(lbl, minlength=k)
    p   = cnt / cnt.sum()
    ent = -np.sum([x*math.log(x+1e-12) for x in p if x > 0])
    return float(ent / (math.log(k)+1e-12)), int(n)


print('[*] Metrics defined')


[*] Metrics defined


## CIFAR-100 Dataloaders

In [10]:
import multiprocessing

CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD  = (0.2675, 0.2565, 0.2761)


def get_cifar100_loaders(data_dir='./data', batch_size=256):
    nw = min(multiprocessing.cpu_count(), 4)

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
    ])

    tr = datasets.CIFAR100(root=data_dir, train=True,  download=True, transform=train_tf)
    te = datasets.CIFAR100(root=data_dir, train=False, download=True, transform=test_tf)

    kw = dict(num_workers=nw, pin_memory=True, persistent_workers=(nw>0))
    trl = DataLoader(tr, batch_size=batch_size, shuffle=True,  **kw)
    tel = DataLoader(te, batch_size=batch_size, shuffle=False, **kw)
    print(f'[*] CIFAR-100 — bs={batch_size} workers={nw} | '
          f'train={len(tr):,} test={len(te):,}')
    return trl, tel


train_loader, test_loader = get_cifar100_loaders(batch_size=256)


100%|██████████| 169M/169M [00:05<00:00, 31.4MB/s]


[*] CIFAR-100 — bs=256 workers=2 | train=50,000 test=10,000


## HookedResNetPGD

In [6]:
class HookedResNetPGD(nn.Module):
    """
    ResNet-18 with L2-PGD latent attack on layer2.
    No spectral constraints — purely unconstrained gradient ascent in feature space.
    """

    def __init__(self, num_classes=100, pgd_eps=2.0,
                 pgd_steps=1, pgd_alpha=1.0):
        """
        pgd_eps   : L2 ball radius in feature space
        pgd_steps : inner steps (1 = fast-AT)
        pgd_alpha : step size per iteration
        """
        super().__init__()
        # ── backbone ────────────────────────────────────────────────────
        self.model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.model.conv1   = nn.Conv2d(3, 64, 3, 1, 1, bias=False)  # 7x7→3x3
        self.model.maxpool = nn.Identity()                            # remove pool
        self.model.fc      = nn.Linear(self.model.fc.in_features, num_classes)

        # ── PGD config ──────────────────────────────────────────────────
        self.pgd_eps   = pgd_eps
        self.pgd_steps = pgd_steps
        self.pgd_alpha = pgd_alpha
        self.apply_pgd = False

        # ── diagnostics (populated every attacked forward pass) ──────────
        self.last_delta_l2  = None   # mean L2 norm of perturbation
        self.last_delta_rel = None   # delta_l2 / feature_l2

    # ── network halves ───────────────────────────────────────────────────
    def _stem_to_layer2(self, x):
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)     # Identity
        x = self.model.layer1(x)
        x = self.model.layer2(x)
        return x                      # (B, 128, H, W)

    def _layer3_to_logits(self, z):
        z = self.model.layer3(z)
        z = self.model.layer4(z)
        z = self.model.avgpool(z)
        return self.model.fc(torch.flatten(z, 1))

    # ── L2 PGD attack ─────────────────────────────────────────────────────
    def _pgd_attack_features(self, z_clean, labels):
        """
        L2-PGD in feature space (Madry et al., Algorithm 1).

        Step: δ ← δ + α · ∇_δ L / ‖∇_δ L‖₂   (per sample)
        Proj: δ ← ε · δ / max(ε, ‖δ‖₂)        (per sample)

        Runs fully in fp32. Do NOT wrap in autocast.
        z_clean must be detached (no gradient through the clean path).
        """
        B = z_clean.shape[0]
        # zero-start: no random initialisation (faster, slightly weaker)
        delta = torch.zeros_like(z_clean)   # leaf tensor, no grad yet

        for _ in range(self.pgd_steps):
            delta = delta.detach().requires_grad_(True)   # fresh leaf each step

            logits = self._layer3_to_logits(z_clean + delta)
            loss   = F.cross_entropy(logits, labels)

            grad = torch.autograd.grad(loss, delta)[0]    # (B,C,H,W)

            # ── L2 normalised step ──────────────────────────────────────
            # Normalise per sample to get unit-gradient direction
            g_norm = grad.flatten(1).norm(p=2, dim=1).clamp(min=1e-8)  # (B,)
            g_unit = grad / g_norm.view(B, 1, 1, 1)     # unit direction
            delta  = delta.detach() + self.pgd_alpha * g_unit

            # ── L2 projection onto eps-ball ─────────────────────────────
            d_norm = delta.flatten(1).norm(p=2, dim=1).clamp(min=1e-8)  # (B,)
            scale  = (self.pgd_eps / d_norm).clamp(max=1.0)
            delta  = delta * scale.view(B, 1, 1, 1)

        # ── store diagnostics ─────────────────────────────────────────────
        with torch.no_grad():
            dl2 = delta.flatten(1).norm(p=2, dim=1).mean()
            cl2 = z_clean.flatten(1).norm(p=2, dim=1).mean()
            self.last_delta_l2  = dl2.item()
            self.last_delta_rel = (dl2 / cl2.clamp(min=1e-8)).item()

        return (z_clean + delta).detach()

    # ── forward ───────────────────────────────────────────────────────────
    def forward(self, x, labels=None):
        """
        Clean forward if apply_pgd=False or labels=None or not training.
        Attacked forward otherwise.
        """
        z = self._stem_to_layer2(x)
        if self.apply_pgd and labels is not None and self.training:
            z = self._pgd_attack_features(z.detach(), labels)
        return self._layer3_to_logits(z)

    def forward_with_features(self, x):
        """Evaluation: returns (logits, penultimate_features). Always clean."""
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)
        x = self.model.layer1(x)
        x = self.model.layer2(x)
        x = self.model.layer3(x)
        x = self.model.layer4(x)
        x = self.model.avgpool(x)
        f = torch.flatten(x, 1)
        return self.model.fc(f), f


# ── sanity check ─────────────────────────────────────────────────────────
_m = HookedResNetPGD(num_classes=100).to(device)
_x = torch.randn(4,3,32,32).to(device)
_y = torch.randint(0,100,(4,)).to(device)
_m.eval();  _m.apply_pgd = False
assert _m(_x).shape       == (4,100), 'clean forward broken'
_m.train(); _m.apply_pgd = True
assert _m(_x,_y).shape    == (4,100), 'pgd forward broken'
_l, _f = _m.forward_with_features(_x)
assert _l.shape == (4,100) and _f.shape[1]==512, 'forward_with_features broken'
del _m,_x,_y,_l,_f
print('[*] All sanity checks passed')


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 82.2MB/s]


NameError: name 'device' is not defined

## Evaluate Model (clean accuracy)

In [ ]:
def evaluate_model(model, loader, device):
    model.eval(); model.apply_pgd = False
    total=correct=0
    confs,flags,feats,preds,trues=[],[],[],[],[]
    with torch.no_grad():
        for imgs,lbls in loader:
            imgs,lbls = imgs.to(device),lbls.to(device)
            logits,f  = model.forward_with_features(imgs)
            pr        = F.softmax(logits,dim=1)
            cf,pd     = pr.max(dim=1)
            cb        = pd.eq(lbls)
            total    += imgs.size(0)
            correct  += cb.sum().item()
            confs.append(cf.cpu().numpy())
            flags.append(cb.cpu().numpy())
            feats.append(f.cpu().numpy())
            preds.append(pd.cpu().numpy())
            trues.append(lbls.cpu().numpy())
    acc = correct/total
    c,fl = np.concatenate(confs),np.concatenate(flags)
    ece  = compute_ece(c,fl)
    return (acc, ece,
            np.concatenate(feats),
            np.concatenate(preds),
            np.concatenate(trues))

print('[*] evaluate_model defined')


[*] evaluate_model defined


## Training Loop

**Two updates per batch** (when adv_active=True):

1. Clean forward → CE loss → backbone update (standard SGD step)
2. PGD-attacked forward → CE loss on attacked features → backbone update

In [ ]:
def train_pgd_epoch(train_loader, model, optimizer, criterion,
                    epoch, device, adv_active, scaler):
    model.train()
    stats = {'loss_cl':[], 'loss_adv':[], 'delta_rel':[], 'loss_inc':[]}
    pbar  = tqdm(train_loader, desc=f'Ep {epoch:3d}', leave=False)

    for imgs, lbls in pbar:
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)

        # ── Step 1: clean update (AMP safe) ────────────────────────────
        model.apply_pgd = False
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type='cuda'):
            loss_cl = criterion(model(imgs), lbls)
        scaler.scale(loss_cl).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()                     # ONE update per step
        stats['loss_cl'].append(loss_cl.item())

        if not adv_active:
            pbar.set_postfix({'cl': f'{loss_cl.item():.3f}', 'adv':'off'})
            continue

        # ── Step 2: adversarial update (full fp32 — no autocast) ───────
        # PGD calls autograd.grad internally. autocast + manual grad = wrong.
        model.apply_pgd = True
        optimizer.zero_grad(set_to_none=True)
        # No autocast here — full fp32
        loss_adv = criterion(model(imgs, lbls), lbls)
        loss_adv.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()                    # plain step, no scaler
        model.apply_pgd = False

        stats['loss_adv'].append(loss_adv.item())
        stats['loss_inc'].append(loss_adv.item() - loss_cl.item())
        if model.last_delta_rel is not None:
            stats['delta_rel'].append(model.last_delta_rel)

        pbar.set_postfix({
            'cl':  f'{loss_cl.item():.3f}',
            'adv': f'{loss_adv.item():.3f}',
            'inc': f'{loss_adv.item()-loss_cl.item():+.3f}',
            'rel': f'{model.last_delta_rel:.3f}' if model.last_delta_rel else '?'
        })

    _m = lambda l: float(sum(l)/len(l)) if l else 0.0
    return {k: _m(v) for k,v in stats.items()}


print('[*] train_pgd_epoch defined')


[*] train_pgd_epoch defined


## Plot and Save History

In [ ]:
def plot_and_save_history(history, save_dir):
    ensure_dir(save_dir)
    fig, axes = plt.subplots(1, 3, figsize=(18,5))

    ax = axes[0]
    ax.plot(history['loss_cl'],  label='Clean Loss',   color='blue')
    ax.plot(history['loss_adv'], label='PGD Adv Loss', color='red')
    ax.set_title('Loss: Clean vs PGD Adversarial')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax2 = ax.twinx()
    ax.plot(history['test_acc'], color='green',  label='Test Acc')
    ax2.plot(history['fds'],     color='purple', label='FDS', linestyle='--')
    ax.set_title('Test Accuracy & FDS'); ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy'); ax2.set_ylabel('FDS')
    ax.legend(loc='upper left'); ax2.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

    ax = axes[2]
    ax.plot(history['delta_rel'], label='Rel Perturbation', color='orange')
    ax.plot(history['loss_inc'],  label='Loss Increase',    color='red',
            linestyle='--')
    ax.set_title('PGD Attack Strength')
    ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    p = os.path.join(save_dir, 'pgd_history.png')
    plt.savefig(p, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'[*] Plot → {p}')

print('[*] plot_and_save_history defined')


[*] plot_and_save_history defined


## Training Run

**Config notes:**
- `pgd_eps=2.0` — L2 radius in layer2 feature space. Layer2 feature norms on
  CIFAR-100 are typically 60–100. eps=2.0 gives ~2–3% relative perturbation,
  enough to be adversarial. Increase to 4.0 if `delta_rel < 0.01` after ep5.
- `pgd_alpha=1.0` — single step, alpha=eps for maximum single-step displacement.
- `adv_start_epoch=5` — at epoch 5, CIFAR-100 clean loss is ~3.0–3.5.
  Generator has real gradient signal. Do not start at 0 (loss too high,
  unstable) or at convergence (loss too low, no signal).
- `lr=0.1` standard SGD, cosine annealing over full 50 epochs.

In [ ]:
def run_pgd_experiment(
        epochs=50,
        save_dir='/content/drive/MyDrive/pgd_latent_runs',
        adv_start_epoch=5,
        pgd_eps=2.0,
        pgd_steps=1,
        pgd_alpha=1.0,
        lr=0.1,
        batch_size=256):
    ensure_dir(save_dir)

    model = HookedResNetPGD(num_classes=100,
                            pgd_eps=pgd_eps,
                            pgd_steps=pgd_steps,
                            pgd_alpha=pgd_alpha).to(device)

    trl, tel = get_cifar100_loaders(batch_size=batch_size)
    crit  = nn.CrossEntropyLoss()
    opt   = optim.SGD(model.parameters(), lr=lr,
                      momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    # Scaler used ONLY for the clean step (adv step is full fp32)
    scaler = torch.cuda.amp.GradScaler()

    best_p   = os.path.join(save_dir, 'best_pgd_ckpt.pth')
    latest_p = os.path.join(save_dir, 'latest_pgd_ckpt.pth')
    hist_p   = os.path.join(save_dir, 'history.json')

    start, best_acc = 0, 0.0
    history = {k:[] for k in ['loss_cl','loss_adv','delta_rel','loss_inc',
                               'test_acc','fds','ece']}

    # ── auto-resume ──────────────────────────────────────────────────────
    for rp in [latest_p, best_p]:
        if os.path.exists(rp):
            print(f'[*] Resuming from {rp}')
            ck = torch.load(rp, map_location=device)
            model.load_state_dict(ck['model_state'])
            opt.load_state_dict(ck['opt_state'])
            sched.load_state_dict(ck['sched_state'])
            scaler.load_state_dict(ck['scaler_state'])
            start, best_acc = ck.get('epoch',0)+1, ck.get('best_acc',0.0)
            if os.path.exists(hist_p) and os.path.getsize(hist_p)>0:
                try:
                    history = json.load(open(hist_p))
                    print(f'    history: {len(history["test_acc"])} epochs')
                except: pass
            break

    print(f'\n{"="*65}')
    print(f' PGD Latent — CIFAR-100 | eps={pgd_eps} steps={pgd_steps} '
          f'alpha={pgd_alpha} adv_start={adv_start_epoch}')
    print(f' inc>0 and rel>0.01 expected after ep {adv_start_epoch}')
    print(f'{"="*65}\n')

    for epoch in range(start, epochs):
        adv = epoch >= adv_start_epoch
        t0  = time.time()

        agg = train_pgd_epoch(trl, model, opt, crit,
                              epoch+1, device, adv, scaler)
        sched.step()

        acc,ece,feats,preds,trues = evaluate_model(model, tel, device)
        fds,_ = compute_failure_diversity(feats, preds, trues)

        for k,v in agg.items():
            if k in history: history[k].append(v)
        history['test_acc'].append(acc)
        history['fds'].append(fds)
        history['ece'].append(ece)

        print(f'Ep {epoch+1:3d}/{epochs} | {time.time()-t0:.0f}s | '
              f'cl={agg["loss_cl"]:.3f} | '
              f'adv={agg["loss_adv"]:.3f} | '
              f'inc={agg["loss_inc"]:+.3f} | '
              f'rel={agg["delta_rel"]:.4f} | '
              f'acc={acc:.4f} | ece={ece:.4f} | '
              f'{"ADV" if adv else "warm"}')

        if adv and agg['loss_inc'] < 0.001 and epoch > adv_start_epoch+5:
            print(f'  ⚠ loss_inc near 0 — model fully adapted. '
                  f'Consider increasing pgd_eps.')
        if adv and agg['delta_rel'] < 0.005 and epoch > adv_start_epoch:
            print(f'  ⚠ delta_rel near 0 — PGD not attacking. '
                  f'Increase pgd_eps.')

        ck = {'epoch':epoch, 'model_state':model.state_dict(),
              'opt_state':opt.state_dict(), 'sched_state':sched.state_dict(),
              'scaler_state':scaler.state_dict(), 'best_acc':best_acc}
        torch.save(ck, latest_p)
        if acc > best_acc:
            best_acc = acc; ck['best_acc'] = best_acc
            torch.save(ck, best_p)
        json.dump(history, open(hist_p,'w'))

    print(f'\n[*] Done. Best acc: {best_acc:.4f}')
    plot_and_save_history(history, save_dir)
    return model, history


pgd_model, pgd_history = run_pgd_experiment(
    epochs=50, save_dir='/content/drive/MyDrive/pgd_latent_runs',
    adv_start_epoch=5, pgd_eps=2.0, pgd_steps=1, pgd_alpha=1.0,
    lr=0.1, batch_size=256
)

/tmp/ipykernel_2288/3490280429.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


[*] CIFAR-100 — bs=256 workers=2 | train=50,000 test=10,000

 PGD Latent — CIFAR-100 | eps=2.0 steps=1 alpha=1.0 adv_start=5
 inc>0 and rel>0.01 expected after ep 5



Ep   1/50 | 28s | cl=2.476 | adv=0.000 | inc=+0.000 | rel=0.0000 | acc=0.5350 | ece=0.0412 | warm


Ep   2/50 | 27s | cl=1.312 | adv=0.000 | inc=+0.000 | rel=0.0000 | acc=0.5722 | ece=0.0755 | warm


Ep   3/50 | 31s | cl=1.048 | adv=0.000 | inc=+0.000 | rel=0.0000 | acc=0.6222 | ece=0.0527 | warm


...

Ep  46/50 | 56s | cl=0.026 | adv=0.636 | inc=+0.610 | rel=0.0653 | acc=0.8100 | ece=0.0167 | ADV


Ep  47/50 | 55s | cl=0.024 | adv=0.615 | inc=+0.591 | rel=0.0656 | acc=0.8110 | ece=0.0190 | ADV


Ep  48/50 | 55s | cl=0.023 | adv=0.602 | inc=+0.580 | rel=0.0657 | acc=0.8105 | ece=0.0189 | ADV


Ep  49/50 | 55s | cl=0.022 | adv=0.595 | inc=+0.572 | rel=0.0658 | acc=0.8091 | ece=0.0170 | ADV


Ep  50/50 | 55s | cl=0.022 | adv=0.590 | inc=+0.568 | rel=0.0658 | acc=0.8109 | ece=0.0190 | ADV

[*] Done. Best acc: 0.8116
[*] Plot → /content/drive/MyDrive/pgd_latent_runs/pgd_history.png


## 11. Download CIFAR-100-C

Downloaded once to Drive. ~11 GB. Zenodo record 3555552.
Handles both possible extraction layouts (subfolder or flat).

In [13]:
def download_and_extract_cifar100c(
        drive_path='/content/drive/MyDrive/CIFAR-100-C_Data'):
    """
    Downloads and extracts CIFAR-100-C if not already in Drive.
    Handles two possible tar layouts:
      Layout A: extracts to drive_path/CIFAR-100-C/labels.npy
      Layout B: extracts flat to drive_path/labels.npy
    Returns the directory containing labels.npy.
    """
    ensure_dir(drive_path)

    # Check both possible locations
    subfolder = os.path.join(drive_path, 'CIFAR-100-C')
    for candidate in [subfolder, drive_path]:
        if os.path.exists(os.path.join(candidate, 'labels.npy')):
            print(f'[*] CIFAR-100-C found at {candidate}')
            return candidate

    tar_path = os.path.join(drive_path, 'CIFAR-100-C.tar')
    print('[*] Downloading CIFAR-100-C')
    url = 'https://zenodo.org/record/3555552/files/CIFAR-100-C.tar?download=1'
    urllib.request.urlretrieve(url, tar_path)
    print('[*] Extracting ...')
    with tarfile.open(tar_path) as tar:
        tar.extractall(path=drive_path)
    print('[*] Extraction complete.')

    # Detect layout after extraction
    for candidate in [subfolder, drive_path]:
        if os.path.exists(os.path.join(candidate, 'labels.npy')):
            return candidate

    raise RuntimeError('CIFAR-100-C extraction failed — labels.npy not found.')


cifar100c_dir = download_and_extract_cifar100c()
print(f'[*] Dataset dir: {cifar100c_dir}')
print('[*] Files:', sorted(os.listdir(cifar100c_dir))[:5], '...')


[*] CIFAR-100-C found at /content/drive/MyDrive/CIFAR-100-C_Data/CIFAR-100-C
[*] Dataset dir: /content/drive/MyDrive/CIFAR-100-C_Data/CIFAR-100-C
[*] Files: ['README.txt', 'brightness.npy', 'contrast.npy', 'defocus_blur.npy', 'elastic_transform.npy'] ...


## CIFAR-100-C Robustness Evaluation

Reports accuracy on all 19 corruption types at severity 5.
MCA = mean over all 19. Primary metric for the paper table.

**Memory fix:** images are loaded to CPU, then sent to GPU in chunks of 500.
Never 10,000 images on GPU at once.

In [ ]:
CORRUPTIONS_19 = [
    'gaussian_noise', 'shot_noise',    'impulse_noise',
    'defocus_blur',   'glass_blur',    'motion_blur',    'zoom_blur',
    'snow',           'frost',          'fog',
    'brightness',     'contrast',
    'elastic_transform','pixelate',    'jpeg_compression',
    'speckle_noise',  'gaussian_blur', 'spatter',        'saturate',
]


def evaluate_robustness_cifar100c(
        model, device, severity=5,
        drive_path='/content/drive/MyDrive/CIFAR-100-C_Data',
        chunk=500):
    """
    Evaluates on CIFAR-100-C.
    Images loaded to CPU, normalised on CPU, sent to GPU in chunks.
    Avoids OOM from loading all 10k images at once.
    """
    print(f'\n{"="*55}')
    print(f' CIFAR-100-C  Severity={severity}')
    print(f'{"="*55}')

    model.eval(); model.apply_pgd = False
    c100c = download_and_extract_cifar100c(drive_path)

    start = (severity-1)*10000
    labels_np = np.load(os.path.join(c100c,'labels.npy'))[start:start+10000]

    # Normalisation tensors on CPU (applied before chunking to GPU)
    mu  = torch.tensor(CIFAR100_MEAN).view(1,3,1,1)   # CPU
    sig = torch.tensor(CIFAR100_STD ).view(1,3,1,1)   # CPU

    results = {}
    with torch.no_grad():
        for c in CORRUPTIONS_19:
            path = os.path.join(c100c, f'{c}.npy')
            if not os.path.exists(path):
                print(f'  {c:<22} MISSING')
                continue

            # Load all on CPU — normalise on CPU — send chunks to GPU
            data = np.load(path)[start:start+10000]   # uint8 (10000,32,32,3)
            imgs = torch.from_numpy(data).permute(0,3,1,2).float().div(255.0)
            imgs = (imgs - mu) / sig                   # normalised, still CPU
            lbl  = torch.from_numpy(labels_np).long()

            correct = 0
            for i in range(0, 10000, chunk):
                out = model(imgs[i:i+chunk].to(device)).argmax(1)
                correct += out.eq(lbl[i:i+chunk].to(device)).sum().item()

            acc = correct / 10000.0
            results[c] = acc
            print(f'  {c:<22}: {acc*100:.2f}%')

    mca = float(np.mean(list(results.values())))
    results['MCA'] = mca
    print(f'\n  MCA ({len(results)-1} corruptions): {mca*100:.2f}%')
    print('='*55)
    return results


# load best checkpoint and evaluate
pgd_model = HookedResNetPGD(num_classes=100).to(device)
ckpt = torch.load('/content/drive/MyDrive/pgd_latent_runs/best_pgd_ckpt.pth',
                  map_location=device)
pgd_model.load_state_dict(ckpt['model_state'])
print(f'[*] Loaded checkpoint  best_acc={ckpt["best_acc"]:.4f}')

rob_results = evaluate_robustness_cifar100c(
    pgd_model, device, severity=5)

with open('/content/drive/MyDrive/pgd_latent_runs/cifar100c_results.json','w') as f:
    json.dump(rob_results, f, indent=2)
print('[*] Saved cifar100c_results.json')


[*] Loaded checkpoint  best_acc=0.8116

 CIFAR-100-C  Severity=5
[*] CIFAR-100-C found at /content/drive/MyDrive/CIFAR-100-C_Data/CIFAR-100-C
  gaussian_noise        : 8.67%
  shot_noise            : 10.38%
  impulse_noise         : 6.26%
  defocus_blur          : 34.30%
  glass_blur            : 18.45%
  motion_blur           : 45.42%
  zoom_blur             : 42.31%
  snow                  : 50.41%
  frost                 : 35.75%
  fog                   : 45.55%
  brightness            : 70.28%
  contrast              : 21.57%
  elastic_transform     : 53.98%
  pixelate              : 24.08%
  jpeg_compression      : 45.92%
  speckle_noise         : 12.46%
  gaussian_blur         : 21.54%
  spatter               : 48.38%
  saturate              : 63.17%

  MCA (19 corruptions): 34.68%
[*] Saved cifar100c_results.json


In [3]:
!pip install grad-cam -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 83.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## PGD Diagnostic Suite

Three experiments that characterise the spectral and semantic behaviour of PGD.

**Exp 1 — Spectral signature:** FFT of `z_adv - z_clean` in layer2 feature space.
LF/HF energy split and phase/amplitude delta. Expected: PGD touches LF bands
and scrambles phase more than amplitude. This contrasts with ALSP (LF≈0, phase≈0).

**Exp 2 — Attribution drift:** GradCAM IoU at layer4[-1] comparing clean vs
PGD-attacked features (injected via temporary hook). vs matched-budget noise baseline.
Expected: PGD IoU < noise baseline = PGD moves attention off-target.

**Exp 3 — Corruption spectral signatures:** FFT of `(corrupted - clean)` input
images, raw [0,1] pixels, NO normalisation. rfftfreq-based distance. Tells us
which corruption types are HF vs LF dominant — explains per-corruption SALA gains.

**Injection approach for Exp2:** Since PGD attacks feature space (not input space),
we cannot just pass a perturbed input to GradCAM. Instead, z_adv is injected
into the forward pass via a temporary hook on layer2 that overrides its output.
The capture hook is removed during injection to avoid stale value in self.captured.

In [4]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from scipy.ndimage import center_of_mass


class PGDDiagnostic:

    def __init__(self, model, device='cuda'):
        self.device = torch.device(device)
        self.model  = model.to(self.device)
        self.model.eval()
        self.model.apply_pgd = False
        self.captured = {}
        # Capture hook on layer2 (where PGD operates)
        self._l2_h = self.model.model.layer2.register_forward_hook(
            lambda m,i,o: self.captured.update({'l2': o.detach()}))

    def cleanup(self):
        self._l2_h.remove()

    # ── Experiment 1 ──────────────────────────────────────────────────────
    def exp1_spectral_signature(self, clean_loader, n_batches=10):
        """
        What frequency bands does PGD perturb?
        Compares layer2 features: ALSP-off vs PGD-attacked.
        Calls _pgd_attack_features directly — no model.train() needed.
        """
        lf_fracs, hf_fracs, ph_ds, amp_ds = [], [], [], []
        loader = iter(clean_loader)
        n_seen = 0

        for _ in range(n_batches):
            try: x, lbls = next(loader)
            except StopIteration: break
            x, lbls = x.to(self.device), lbls.to(self.device)

            # z_clean — no PGD
            self.model.apply_pgd = False
            with torch.no_grad(): self.model(x)
            z_c = self.captured['l2'].clone()

            # z_adv — direct call, model stays in eval mode
            # _pgd_attack_features only needs autograd, not training mode
            z_a = self.model._pgd_attack_features(z_c.detach(), lbls)

            delta   = z_a - z_c
            B,C,H,W = delta.shape
            Fd      = fft_module.rfft2(delta, norm='ortho')
            power   = Fd.abs()**2

            fy = fft_module.fftfreq(H, device=self.device)
            fx = fft_module.rfftfreq(W, device=self.device)
            FY,FX = torch.meshgrid(fy, fx, indexing='ij')
            dist  = torch.sqrt(FY**2 + FX**2)
            lf_m  = (dist <= 0.15).float()
            hf_m  = (dist >  0.15).float()
            tot   = power.sum() + 1e-8
            lf_fracs.append(((power*lf_m).sum()/tot).item())
            hf_fracs.append(((power*hf_m).sum()/tot).item())

            Fc = fft_module.rfft2(z_c, norm='ortho')
            Fa = fft_module.rfft2(z_a, norm='ortho')
            amp_ds.append((Fa.abs()-Fc.abs()).abs().mean().item())
            pr  = Fa.angle() - Fc.angle()
            ph_ds.append(torch.atan2(torch.sin(pr),
                                     torch.cos(pr)).abs().mean().item())
            n_seen += B

        r = {
            'lf_frac':   float(np.mean(lf_fracs)),
            'hf_frac':   float(np.mean(hf_fracs)),
            'phase_d':   float(np.mean(ph_ds)),
            'amp_d':     float(np.mean(amp_ds)),
            'ph_amp_r':  float(np.mean(ph_ds))/(float(np.mean(amp_ds))+1e-8),
            'n':         n_seen,
        }
        print('\n'+'='*62)
        print(' EXP 1: PGD Spectral Signature  (eval mode, direct call)')
        print('='*62)
        print(f'  n={r["n"]}  LF={r["lf_frac"]:.4f}  HF={r["hf_frac"]:.4f}')
        print(f'  phase_delta={r["phase_d"]:.6f} rad  amp_delta={r["amp_d"]:.6f}')
        print(f'  phase/amp ratio = {r["ph_amp_r"]:.2f}x')
        print()
        if r['lf_frac'] > 0.10:
            print(f'  ⚠  {r["lf_frac"]*100:.1f}% energy in LF — PGD touches semantic bands.')
            print('     → Confirms need for SALA LF mask.')
        else:
            print('  ✓  Energy in HF — check phase/amp ratio for phase scrambling.')
        if r['ph_amp_r'] > 2.0:
            print(f'  ⚠  Phase {r["ph_amp_r"]:.1f}x more scrambled than amplitude.')
            print('     → Confirms need for SALA phase-lock.')
        elif r['ph_amp_r'] > 1.0:
            print(f'  ⚠  Phase delta > amp delta ({r["ph_amp_r"]:.1f}x). Moderate scrambling.')
        print('='*62)
        return r


    # ── Experiment 2 ──────────────────────────────────────────────────────
    def exp2_attribution_drift(self, clean_loader, n_samples=200):
        """
        Three-way attribution drift comparison at layer4[-1].

          PGD   — adversarial attack in feature space (our method under study)
          FeatN — gaussian noise injected into FEATURE space at matched L2 norm
                  (Gemini's fix: apples-to-apples space comparison)
          InpN  — gaussian noise injected into INPUT space at CIFAR-C severity-5
                  level (real-world reference — what actually happens in deployment)

        If PGD < FeatN: the adversarial direction specifically targets semantics,
                        not just any perturbation of that magnitude.
        If PGD < InpN:  latent PGD is more harmful than real-world corruption.
        Both being true = strongest justification for spectral constraints.

        Injection: z_adv and z_feat_noised both injected via temporary hook.
        Capture hook removed during injection to avoid stale captured value,
        then re-registered after each injection.
        """
        cam = GradCAM(model=self.model.model,
                      target_layers=[self.model.model.layer4[-1]])

        # CIFAR-C severity 5 gaussian noise in normalised pixel space.
        # CIFAR-100 normalised std ≈ 0.267. CIFAR-C sev5 gaussian noise
        # has std=62/255≈0.243 in [0,1] → 0.243/0.267≈0.91 in normalised space.
        # Use 0.5 as a conservative estimate that is visually comparable to sev5.
        CIFARC_SEV5_STD = 0.5

        pm  = {'iou':[],'cos':[],'com':[]}   # PGD (adversarial, feature space)
        fnm = {'iou':[],'cos':[],'com':[]}   # Feature-space noise (Gemini fix)
        inm = {'iou':[],'cos':[],'com':[]}   # Input-space noise (real-world ref)
        done = 0

        for xb, lb in clean_loader:
            if done >= n_samples: break
            xb = xb.to(self.device)
            lb = lb.to(self.device)

            for i in range(xb.size(0)):
                if done >= n_samples: break
                x1   = xb[i:i+1]
                tgts = [ClassifierOutputTarget(lb[i].item())]

                # ── clean GradCAM ─────────────────────────────────────────
                self.model.apply_pgd = False
                gc = cam(input_tensor=x1, targets=tgts)[0]

                # ── get z_clean and z_adv ─────────────────────────────────
                self.model.apply_pgd = False
                with torch.no_grad(): self.model(x1)
                z_c = self.captured['l2'].clone()
                z_a = self.model._pgd_attack_features(z_c.detach(), lb[i:i+1])

                # exact L2 norm of PGD perturbation (per sample, scalar here B=1)
                pgd_l2 = (z_a - z_c).flatten().norm(p=2).item()

                # ── helper: inject z into layer2 and get GradCAM ──────────
                def _cam_with_injection(z_inject):
                    self._l2_h.remove()
                    _s = [z_inject]
                    h  = self.model.model.layer2.register_forward_hook(
                             lambda m, inp, out, s=_s: s[0])
                    self.model.apply_pgd = False
                    g  = cam(input_tensor=x1, targets=tgts)[0]
                    h.remove()
                    # re-register capture hook
                    self._l2_h = self.model.model.layer2.register_forward_hook(
                        lambda m, i, o: self.captured.update({'l2': o.detach()}))
                    return g

                # ── PGD GradCAM ───────────────────────────────────────────
                gp = _cam_with_injection(z_a)
                pm['iou'].append(self._iou(gc, gp))
                pm['cos'].append(self._cos(gc, gp))
                pm['com'].append(self._com(gc, gp))

                # ── Feature-space noise at matched L2 (Gemini fix) ────────
                # Random unit direction in feature space, scaled to pgd_l2
                noise_f  = torch.randn_like(z_c)
                noise_f  = noise_f / noise_f.flatten().norm(p=2).clamp(min=1e-8)
                noise_f  = noise_f * pgd_l2          # exact same L2 as PGD
                z_fn     = (z_c + noise_f).detach()
                gfn      = _cam_with_injection(z_fn)
                fnm['iou'].append(self._iou(gc, gfn))
                fnm['cos'].append(self._cos(gc, gfn))
                fnm['com'].append(self._com(gc, gfn))

                # ── Input-space noise at CIFAR-C severity-5 level ─────────
                # This is the real-world reference: what a physical corruption
                # does to attribution, not a controlled feature-space perturbation.
                xn  = (x1 + torch.randn_like(x1) * CIFARC_SEV5_STD).clamp(-3, 3)
                self.model.apply_pgd = False
                gin = cam(input_tensor=xn, targets=tgts)[0]
                inm['iou'].append(self._iou(gc, gin))
                inm['cos'].append(self._cos(gc, gin))
                inm['com'].append(self._com(gc, gin))

                done += 1

        r  = {k:float(np.mean(v)) for k,v in pm.items()}
        fn = {k:float(np.mean(v)) for k,v in fnm.items()}
        ip = {k:float(np.mean(v)) for k,v in inm.items()}

        print('\n'+'='*72)
        print(' EXP 2: Attribution Drift — layer4[-1] — three-way comparison')
        print(f' n={done}')
        print('='*72)
        print(f'  {"Metric":<26} {"PGD":>8} {"FeatNoise":>10} {"InpNoise":>10}')
        print('-'*72)
        for k,lbl,up in [
            ('iou','GradCAM IoU  (↑)',True),
            ('cos','Cosine Sim   (↑)',True),
            ('com','CoM Shift px (↓)',False),
        ]:
            wf = (r[k]<fn[k]-0.05) if up else (r[k]>fn[k]+0.5)
            wi = (r[k]<ip[k]-0.05) if up else (r[k]>ip[k]+0.5)
            print(f'  {lbl:<26} {r[k]:>8.4f} {fn[k]:>10.4f} {ip[k]:>10.4f}  '
                  f'{"⚠vsF" if wf else "✓F"} {"⚠vsI" if wi else "✓I"}')

        print()
        if r['iou'] < fn['iou'] - 0.05:
            print('  ⚠  PGD worse than RANDOM feature noise at same L2 budget.')
            print('     Adversarial direction specifically targets semantic features.')
            print('     Strongest possible justification for spectral constraints.')
        else:
            print('  ✓  PGD comparable to random feature noise at same L2.')
            print('     Damage from magnitude, not adversarial direction.')
            print('     Spectral justification via Exp1 LF fraction still holds.')
        if r['iou'] < ip['iou'] - 0.05:
            print('  ⚠  PGD worse than real-world input noise (CIFAR-C sev5).')
            print('     Unconstrained latent attack more harmful than physical corruption.')
        print('='*72)
        return r, fn, ip

    # ── Experiment 3 ──────────────────────────────────────────────────────
    def exp3_corruption_spectral_signatures(
            self, cifar100c_dir, severity=5, n_images=1000):
        """
        LF vs HF energy of each CIFAR-100-C corruption.
        Raw [0,1] pixels — no normalisation (normalisation removes DC, kills LF).
        rfftfreq-based radial distance. Cutoff 0.1 cy/px = periods >10px.
        """
        raw_tf = transforms.Compose([transforms.ToTensor()])
        cds    = datasets.CIFAR100(root='./data', train=False,
                                    download=True, transform=raw_tf)
        clean  = torch.stack([cds[i][0] for i in range(n_images)])
        clean  = clean.to(self.device)

        corrs  = ['gaussian_noise','shot_noise','impulse_noise',
                  'defocus_blur','motion_blur','zoom_blur',
                  'snow','frost','fog','brightness','contrast',
                  'jpeg_compression','pixelate','elastic_transform']
        start  = (severity-1)*10000

        print('\n'+'='*70)
        print(' EXP 3: Corruption Spectral Signatures (raw [0,1], no normalisation)')
        print(f' cutoff=0.1 cy/px | sev={severity} | n={n_images}')
        print('='*70)
        print(f'  {"Corruption":<22} {"LF":>8} {"HF":>8}  Prediction')
        print('-'*70)

        results = {}
        for c in corrs:
            p = os.path.join(cifar100c_dir, f'{c}.npy')
            if not os.path.exists(p):
                print(f'  {c:<22} NOT FOUND')
                continue
            raw  = np.load(p)[start:start+n_images]
            corr = torch.from_numpy(raw).permute(0,3,1,2).float().div(255)
            corr = corr.to(self.device)

            d    = corr - clean
            g    = d.mean(dim=1, keepdim=True)   # greyscale
            F    = fft_module.rfft2(g, norm='ortho')
            pw   = F.abs()**2

            H,Wh = pw.shape[-2], pw.shape[-1]
            W    = (Wh-1)*2
            fy   = fft_module.fftfreq(H, device=self.device)
            fx   = fft_module.rfftfreq(W, device=self.device)
            FY,FX= torch.meshgrid(fy, fx, indexing='ij')
            dist = torch.sqrt(FY**2+FX**2)
            lm   = (dist<=0.1).float()
            hm   = (dist> 0.1).float()
            tot  = pw.sum(dim=(-2,-1),keepdim=True)+1e-8
            lf   = ((pw*lm)/tot).sum(dim=(-2,-1)).mean().item()
            hf   = ((pw*hm)/tot).sum(dim=(-2,-1)).mean().item()

            v = ('HF → SALA helps'   if hf > lf else
                 'LF → SALA limited' if lf > hf*1.5 else
                 'Mixed')
            print(f'  {c:<22} {lf:>8.4f} {hf:>8.4f}  {v}')
            results[c] = {'lf':lf, 'hf':hf}

        print('='*70)
        print('  Use to explain per-corruption MCA gap in paper Table 2.')
        print('='*70)
        return results


    # ── helpers ───────────────────────────────────────────────────────────
    def _iou(self,a,b,p=85):
        ma=a>np.percentile(a,p); mb=b>np.percentile(b,p)
        return float(np.logical_and(ma,mb).sum()/(np.logical_or(ma,mb).sum()+1e-8))

    def _cos(self,a,b):
        fa,fb=a.flatten(),b.flatten()
        return float(np.dot(fa,fb)/(np.linalg.norm(fa)*np.linalg.norm(fb)+1e-8))

    def _com(self,a,b):
        ca=center_of_mass(a) if a.sum()>0 else (0.,0.)
        cb=center_of_mass(b) if b.sum()>0 else (0.,0.)
        return float(np.sqrt((ca[0]-cb[0])**2+(ca[1]-cb[1])**2))


print('[*] PGDDiagnostic defined')


[*] PGDDiagnostic defined


## Run All Diagnostics

In [14]:
# Load best checkpoint
pgd_model = HookedResNetPGD(num_classes=100).to(device)
ckpt = torch.load('/content/drive/MyDrive/pgd_latent_runs/best_pgd_ckpt.pth',
                   map_location=device)
pgd_model.load_state_dict(ckpt['model_state'])
pgd_model.eval()
print(f'[*] Loaded  best_acc={ckpt["best_acc"]:.4f}')

# Diagnostic loader — normalised, no shuffle, 200 samples
diag_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])
diag_ds  = datasets.CIFAR100(root='./data', train=False,
                              download=True, transform=diag_tf)
diag_ldr = DataLoader(Subset(diag_ds, list(range(200))),
                       batch_size=32, shuffle=False)

runner = PGDDiagnostic(model=pgd_model, device=device)

r1 = runner.exp1_spectral_signature(diag_ldr, n_batches=10)
# Fixed: Unpack 3 values instead of 2
r2_pgd, r2_feat, r2_inp = runner.exp2_attribution_drift(diag_ldr, n_samples=200)

cifar100c_dir = download_and_extract_cifar100c()
r3 = runner.exp3_corruption_spectral_signatures(
                    cifar100c_dir, severity=5)

runner.cleanup()

# Updated dict to store all three attribution results
diag = {'exp1': r1, 'exp2_pgd': r2_pgd, 'exp2_base': r2_feat, 'exp2_inp': r2_inp, 'exp3': r3}
with open('/content/drive/MyDrive/pgd_latent_runs/diagnostics.json','w') as f:
    json.dump(diag, f, indent=2)
print('\n[*] Diagnostics complete and saved.')

[*] Loaded  best_acc=0.8116

 EXP 1: PGD Spectral Signature  (eval mode, direct call)
  n=200  LF=0.6458  HF=0.3542
  phase_delta=0.059254 rad  amp_delta=0.002385
  phase/amp ratio = 24.84x

  ⚠  64.6% energy in LF — PGD touches semantic bands.
     → Confirms need for SALA LF mask.
  ⚠  Phase 24.8x more scrambled than amplitude.
     → Confirms need for SALA phase-lock.

 EXP 2: Attribution Drift — layer4[-1] — three-way comparison
 n=200
  Metric                          PGD  FeatNoise   InpNoise
------------------------------------------------------------------------
  GradCAM IoU  (↑)             0.5302     0.9901     0.1535  ⚠vsF ✓I
  Cosine Sim   (↑)             0.9509     1.0000     0.6030  ✓F ✓I
  CoM Shift px (↓)             1.8227     0.0270     9.0739  ⚠vsF ✓I

  ⚠  PGD worse than RANDOM feature noise at same L2 budget.
     Adversarial direction specifically targets semantic features.
     Strongest possible justification for spectral constraints.
[*] CIFAR-100-C found at /

In [ ]:
# Load fresh from disk — safe if run in new session
ckpt = torch.load('/content/drive/MyDrive/pgd_latent_runs/best_pgd_ckpt.pth',
                   map_location='cpu')
rob  = json.load(open('/content/drive/MyDrive/pgd_latent_runs/cifar100c_results.json'))
diag = json.load(open('/content/drive/MyDrive/pgd_latent_runs/diagnostics.json'))

print('='*72)
print(' PAPER TABLE ROW — PGD Latent Baseline')
print(' Method: unconstrained L2-PGD on layer2 features | ResNet-18 | CIFAR-100')
print('='*72)
print(f'  Clean Acc:                {ckpt["best_acc"]*100:.2f}%')
print(f'  MCA (CIFAR-100-C Sev5):   {rob["MCA"]*100:.2f}%')
print()
print(f'  LF energy fraction:       {diag["exp1"]["lf_frac"]:.4f}')
print(f'  HF energy fraction:       {diag["exp1"]["hf_frac"]:.4f}')
print(f'  Phase delta (rad):        {diag["exp1"]["phase_d"]:.6f}')
print(f'  Phase/Amp ratio:          {diag["exp1"]["ph_amp_r"]:.2f}x')
print()
print(f'  GradCAM IoU  PGD:         {diag["exp2_pgd"]["iou"]:.4f}')
print(f'  GradCAM IoU  Noise:       {diag["exp2_base"]["iou"]:.4f}')
print(f'  CoM Shift    PGD (px):    {diag["exp2_pgd"]["com"]:.4f}')
print(f'  CoM Shift    Noise (px):  {diag["exp2_base"]["com"]:.4f}')
print()
print('  → Compare this row to ALSP and SALA rows for Table 1.')
print('='*72)


 PAPER TABLE ROW — PGD Latent Baseline
 Method: unconstrained L2-PGD on layer2 features | ResNet-18 | CIFAR-100
  Clean Acc:                81.16%
  MCA (CIFAR-100-C Sev5):   34.68%

  LF energy fraction:       0.6458
  HF energy fraction:       0.3542
  Phase delta (rad):        0.059254
  Phase/Amp ratio:          24.84x

  GradCAM IoU  PGD:         0.5302
  GradCAM IoU  Noise:       0.9121
  CoM Shift    PGD (px):    1.8227
  CoM Shift    Noise (px):  0.2750

  → Compare this row to ALSP and SALA rows for Table 1.
